In [0]:
%sql

-- PIPELINE: BRONZE -> SILVER
MERGE INTO catalog_ventas.silver.ventas_clean AS target
USING (

  WITH datos_limpios AS (
      SELECT *
      FROM ventas_bronze_EDA
      WHERE precio IS NOT NULL
        AND precio > 0 AND precio <> 0.1
        AND total > 0 AND total <> 0.1
        AND articulo IS NOT NULL AND articulo <> 0
        AND descrip IS NOT NULL
  ),
  datos_duplicados AS (
      SELECT *,
             ROW_NUMBER() OVER (
                 PARTITION BY venta, articulo, descrip, precio
                 ORDER BY vtafecha DESC
             ) AS rn,
              md5(
               concat_ws('||', venta, articulo, descrip, categoria, precio, cant, total, usulogin, cliente, vtaestado, vtafecha, turno, medio_pago, delivery)
           ) AS hash_md5
      FROM datos_limpios
  )
  SELECT
      venta,
      articulo,
      descrip,
      categoria,
      precio,
      cant,
      total,
      usulogin,
      cliente,
      vtaestado,
      vtafecha,
      turno,
      medio_pago,
      delivery,
      hash_md5,
      'bronze_EDA' AS _source_table,
      CURRENT_TIMESTAMP AS _processing_timestamp
  FROM datos_duplicados
  WHERE rn = 1

) AS source

ON target.venta = source.venta
   AND target.hash_md5 <> source.hash_md5

WHEN MATCHED THEN UPDATE SET
  target.articulo              = source.articulo,
  target.descrip               = source.descrip,
  target.categoria             = source.categoria,
  target.precio                = source.precio,
  target.cant                  = source.cant,
  target.total                 = source.total,
  target.usulogin              = source.usulogin,
  target.cliente               = source.cliente,
  target.vtaestado             = source.vtaestado,
  target.vtafecha              = source.vtafecha,
  target.turno                 = source.turno,
  target.medio_pago            = source.medio_pago,           
  target.delivery              = source.delivery,
  target._source_table         = source._source_table,
  target._processing_timestamp = source._processing_timestamp

WHEN NOT MATCHED THEN INSERT  (
  venta,
  articulo,
  descrip,
  categoria,
  precio,
  cant,
  total,
  usulogin,
  cliente,
  vtaestado,
  vtafecha,
  turno,
  medio_pago,
  delivery,
  _source_table,
  _processing_timestamp,
  hash_md5
)
VALUES(
  source.venta,
  source.articulo,
  source.descrip,
  source.categoria,
  source.precio,
  source.cant,
  source.total,
  source.usulogin,
  source.cliente,
  source.vtaestado,
  source.vtafecha,
  source.turno,
  source.medio_pago,
  source.delivery,
  source._source_table,
  source._processing_timestamp,
  source.hash_md5
)




In [0]:
%sql
--COMPARAR REGISTROS BRONZE VS SILVER 
SELECT 
    'Bronze (raw)' as capa,
    COUNT(*) as registros
FROM ventas_bronze_eda

UNION ALL

SELECT 
    'Silver' as capa,
    COUNT(*) as registros
FROM catalog_ventas.silver.ventas_clean;

In [0]:
%sql
--RESUMEN DE LA LIMPIEZA
SELECT
  'ventas' AS tabla,
  56839 AS bronze_registros,
  49081 AS silver_registros,
  7758 AS registros_descartados,
  ROUND(7758 / 56839 * 100, 2) AS porcentaje_limpieza